# 📖 Notebook 1: Hash-Based Sharding

Hash-based sharding is the **default and most common** sharding strategy. You take a shard key (like `user_id`), run it through a hash function, and use modulo to pick which shard the data goes to.

```
shard = hash(user_id) % number_of_shards
```

This gives you **even distribution** — data spreads evenly across all shards regardless of the input pattern.

## Learning Objectives

By the end of this notebook, you'll understand:
- How hash-based sharding distributes data
- How to build a shard router in Python
- How to write and read from the correct shard
- Why adding shards is painful with simple modulo hashing

## 🛠️ Setup

Start the infrastructure first:

```bash
cd core-concepts/sharding
docker-compose up -d
```

### Visualization

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Connect to any shard: Server `shard-1` (or `shard-2`, `shard-3`), User `demo`, Password `demo`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import hashlib
import time

# Each shard is an independent Postgres instance.
# In production these would be separate machines; here they're Docker containers.
SHARD_CONFIGS = {
    0: {"host": "localhost", "port": 5433, "database": "shard_1", "user": "demo", "password": "demo"},
    1: {"host": "localhost", "port": 5434, "database": "shard_2", "user": "demo", "password": "demo"},
    2: {"host": "localhost", "port": 5435, "database": "shard_3", "user": "demo", "password": "demo"},
}

NUM_SHARDS = len(SHARD_CONFIGS)

def get_connection(shard_id):
    """Get a database connection for a specific shard."""
    return psycopg2.connect(**SHARD_CONFIGS[shard_id])

# Quick check — can we connect to all three shards?
for shard_id in SHARD_CONFIGS:
    conn = get_connection(shard_id)
    cur = conn.cursor()
    cur.execute("SELECT current_database()")
    db_name = cur.fetchone()[0]
    print(f"✅ Shard {shard_id} → connected to {db_name}")
    cur.close()
    conn.close()

## Step 1: The Hash Function

The core idea is simple: take the shard key, hash it, and use modulo to pick a shard.

```
hash("user_42") = 123456789
123456789 % 3 = 0  → Shard 0
```

We use a **deterministic** hash so the same key always maps to the same shard. Python's built-in `hash()` is randomized per process, so we use `hashlib` instead.

In [ ]:
def hash_shard(key, num_shards):
    """
    Determine which shard a key belongs to using hash-based sharding.
    
    1. Convert the key to bytes
    2. Hash it with MD5 (fast, good distribution — not for security)
    3. Convert first 8 bytes to an integer
    4. Modulo by number of shards
    """
    key_bytes = str(key).encode('utf-8')
    hash_digest = hashlib.md5(key_bytes).hexdigest()
    hash_int = int(hash_digest[:8], 16)  # first 8 hex chars → integer
    return hash_int % num_shards

# Let's see where different user IDs land
print("User ID → Shard")
print("─" * 25)
for user_id in range(1, 16):
    shard = hash_shard(user_id, NUM_SHARDS)
    print(f"User {user_id:3d}  → Shard {shard}")

Notice how the users **don't** go in order (1→0, 2→1, 3→2, 4→0...). The hash function scrambles the IDs so the distribution is essentially random. That's the whole point — it prevents any pattern in the IDs from creating uneven shards.

## Step 2: Build a Shard Router

A shard router sits between your application and the database shards. It decides which shard to send each query to. Let's build one.

In [ ]:
class ShardRouter:
    """Routes database operations to the correct shard based on hash of the key."""
    
    def __init__(self, shard_configs):
        self.shard_configs = shard_configs
        self.num_shards = len(shard_configs)
    
    def get_shard(self, key):
        """Determine which shard a key belongs to."""
        return hash_shard(key, self.num_shards)
    
    def get_connection(self, shard_id):
        """Get a database connection for a specific shard."""
        return psycopg2.connect(**self.shard_configs[shard_id])
    
    def execute_on_shard(self, key, query, params=None):
        """Execute a query on the shard that owns the given key."""
        shard_id = self.get_shard(key)
        conn = self.get_connection(shard_id)
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(query, params)
        result = None
        if cur.description:  # SELECT query
            result = cur.fetchall()
        cur.close()
        conn.close()
        return shard_id, result
    
    def execute_on_all_shards(self, query, params=None):
        """Execute a query on ALL shards and combine results (scatter-gather)."""
        all_results = []
        for shard_id in range(self.num_shards):
            conn = self.get_connection(shard_id)
            conn.autocommit = True
            cur = conn.cursor()
            cur.execute(query, params)
            if cur.description:
                rows = cur.fetchall()
                all_results.extend(rows)
            cur.close()
            conn.close()
        return all_results

router = ShardRouter(SHARD_CONFIGS)
print(f"✅ Shard router ready with {router.num_shards} shards")

## Step 3: Insert Data Across Shards

Now let's insert 300 users. Each user gets routed to a shard based on their `user_id`. We first clear old data, then insert.

In [ ]:
import random

# Clear existing data from all shards
for shard_id in range(NUM_SHARDS):
    conn = get_connection(shard_id)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("DELETE FROM orders")
    cur.execute("DELETE FROM users")
    cur.close()
    conn.close()

COUNTRIES = ['US', 'UK', 'Germany', 'Japan', 'Brazil', 'India', 'Canada', 'France']

# Insert 300 users across shards
shard_counts = {i: 0 for i in range(NUM_SHARDS)}

for user_id in range(1, 301):
    shard_id = router.get_shard(user_id)
    shard_counts[shard_id] += 1
    
    conn = router.get_connection(shard_id)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO users (id, username, email, country) VALUES (%s, %s, %s, %s)",
        (user_id, f"user_{user_id}", f"user{user_id}@example.com", random.choice(COUNTRIES))
    )
    cur.close()
    conn.close()

print("Users per shard:")
print("─" * 30)
for shard_id, count in shard_counts.items():
    bar = '█' * (count // 3)
    print(f"Shard {shard_id}: {count:3d} users {bar}")
print(f"\nTotal: {sum(shard_counts.values())} users")

**The distribution should be roughly even** — around 100 users per shard. Hash functions aren't perfectly uniform with small datasets, but the distribution gets better as you add more data.

## Step 4: Query a Single Shard (Fast Path)

The best queries hit **one shard only**. If you know the shard key, the router sends the query directly to the right shard. No other shards are involved.

In [ ]:
# Look up a specific user — only hits one shard
target_user_id = 42

start = time.time()
shard_id, result = router.execute_on_shard(
    target_user_id,
    "SELECT id, username, email, country FROM users WHERE id = %s",
    (target_user_id,)
)
elapsed = (time.time() - start) * 1000

print(f"🔍 Looking up user {target_user_id}")
print(f"   Routed to: Shard {shard_id}")
print(f"   Result: {result[0] if result else 'not found'}")
print(f"   Time: {elapsed:.1f} ms")
print()
print("This is fast because we only talked to ONE database, not all three.")

## Step 5: Cross-Shard Query (Scatter-Gather)

What if you need data from **all** users? For example, "how many users are in each country?" Since users are spread across all shards, we have to query every shard and combine the results ourselves. This is called **scatter-gather**.

In [ ]:
# Cross-shard query: count users per country across ALL shards
start = time.time()

# Scatter: query each shard for its country counts
country_totals = {}
for shard_id in range(NUM_SHARDS):
    conn = get_connection(shard_id)
    cur = conn.cursor()
    cur.execute("SELECT country, COUNT(*) FROM users GROUP BY country")
    for country, count in cur.fetchall():
        country_totals[country] = country_totals.get(country, 0) + count
    cur.close()
    conn.close()

elapsed = (time.time() - start) * 1000

# Gather: combine the results
print("🌍 Users per country (cross-shard query):")
print("─" * 30)
for country, count in sorted(country_totals.items(), key=lambda x: -x[1]):
    print(f"  {country:10s} {count:3d}")
print(f"\n⏱️ Time: {elapsed:.1f} ms (had to query all {NUM_SHARDS} shards)")
print()
print("This is slower because we queried EVERY shard and aggregated manually.")
print("In a real system with 64 shards, this would be 64x the network calls.")

## Step 6: The Resharding Problem 💥

Here's the big downside of simple `hash % N` sharding. What happens when you add a 4th shard?

The modulo changes from `% 3` to `% 4`, so **most keys map to a different shard**. Let's see how bad it is.

In [ ]:
# Simulate: how many users would need to move if we add a 4th shard?
moved = 0
stayed = 0

print("What happens when we go from 3 shards to 4 shards?")
print("─" * 50)

for user_id in range(1, 301):
    old_shard = hash_shard(user_id, 3)  # original assignment
    new_shard = hash_shard(user_id, 4)  # new assignment with 4 shards
    if old_shard != new_shard:
        moved += 1
    else:
        stayed += 1

print(f"Users that stay on the same shard: {stayed} ({stayed/300*100:.0f}%)")
print(f"Users that need to MOVE:           {moved} ({moved/300*100:.0f}%)")
print()
print("⚠️  With simple modulo hashing, adding one shard means moving ~75% of data!")
print("   This is why consistent hashing was invented (see Notebook 3).")

## Step 7: Choosing a Good Shard Key

The shard key you pick has a **huge** impact on performance. Let's compare good vs bad shard keys.

In [ ]:
# GOOD shard key: user_id (high cardinality, even distribution)
print("🟢 Shard key: user_id (GOOD)")
print("─" * 40)
good_counts = {i: 0 for i in range(3)}
for uid in range(1, 10001):
    good_counts[hash_shard(uid, 3)] += 1
for s, c in good_counts.items():
    pct = c / 10000 * 100
    bar = '█' * int(pct / 2)
    print(f"  Shard {s}: {c:5d} ({pct:.1f}%) {bar}")

print()

# BAD shard key: is_premium (boolean — only 2 possible values!)
print("🔴 Shard key: is_premium (BAD)")
print("─" * 40)
bad_counts = {i: 0 for i in range(3)}
for uid in range(1, 10001):
    is_premium = uid % 20 == 0  # 5% of users are premium
    bad_counts[hash_shard(is_premium, 3)] += 1
for s, c in bad_counts.items():
    pct = c / 10000 * 100
    bar = '█' * int(pct / 2)
    print(f"  Shard {s}: {c:5d} ({pct:.1f}%) {bar}")
print()
print("Boolean keys only produce 2 unique hashes — one shard is always empty!")
print("This defeats the purpose of sharding.")

## 🎯 Key Takeaways

1. **Hash-based sharding** distributes data evenly using `hash(key) % num_shards`
2. **Single-shard queries are fast** — you only talk to one database
3. **Cross-shard queries are expensive** — you must query all shards and merge results
4. **The resharding problem**: changing `num_shards` moves most of the data
5. **Pick high-cardinality shard keys** — boolean or low-cardinality keys create lopsided shards

### Next Up

**Notebook 2: Range-Based Sharding** — An alternative strategy that splits data by value ranges instead of hashes.